In [1]:
import platform
import subprocess

import cartopy.crs as ccrs
import dask
import matplotlib.pyplot as plt
import metpy
from netCDF4 import Dataset
import numpy as np
import pandas as pd
import scipy as sp
import xarray

# Model Variables
In the following cell you can set the values of the variables relevant to the model. The details of each variable are included in the README. In most cases it is only necessary to set values for the standard variables. Note that any variable included in the model should be given the same value in the postprocess. For example, if the model used zw = 42 and kmax = 11, you should use zw = 42 and kmax = 11 below.

In [2]:
# Set postprocess parameters.

# Standard Variables
zw = 63
kmax = 26
expname='T63L26_ANA'
#DataSetname = 'vvel'
#Dataname = 'v'
#DataSetname = 'uvel'
#Dataname = 'u'
DataSetname = 'geo'
Dataname = 'geo'

#dayst = 4800
dayst=720

# Advanced Variables
imax = None
jmax = None
#imax = 192
#jmax = 96
custom_path = None
custom_kmax = None

In [3]:
# Set Dependent Variables

# Set value of kmax if custom_kmax is used.
if not(custom_kmax is None):
    kmax = custom_kmax
    print("Using custom value for kmax:", kmax)
# Otherwise check value for kmax.
elif kmax!=11 and kmax!=26:
    raise Exception("Unexpected value for kmax. Use custom_kmax and note that other values are implementable, but the user must modify subs1_utils.py routine bscst. If unclear email bkirtman@miami.edu for clarification.")

# Check value for zw.
# Afterwards, set jmax and imax values based on the value given to zw.
# If a value is already given for one of the listed variables, use that instead
match zw:
    case 42:
        jmax = 64 if (jmax is None) else jmax
        imax = 128 if (imax is None) else imax
    case 63:
        jmax = 96 if (jmax is None) else jmax
        imax = 192 if (imax is None) else imax
    case 124:
        jmax = 188 if (jmax is None) else jmax
        imax = 376 if (imax is None) else imax
    case _:
        if (jmax is None) or (imax is None):
            raise Exception("Unexpected value for zw. Other values are implementable, but the user must specify values for jmax and imax in the advanced variables section.")

print("zw =", zw,
      "\nkmax =", kmax,
      "\njmax =", jmax,
      "\nimax =", imax,
      "\ndayst =", dayst)

zw = 63 
kmax = 26 
jmax = 96 
imax = 192 
dayst = 720


In [4]:
# Set datapath.

# If custom_path was set, use that as the datapath.
# Otherwise create an appropriate datapath for the user's operating system.
user_platform = platform.system() if (custom_path is None) else "Custom Path"
print("Setting output datapath for", user_platform)
datapath = ''
match user_platform:
    case 'Custom Path':
        datapath = custom_path
    case 'Windows':
        foo = str(subprocess.check_output(['whoami']))
        end = len(foo) - 5
        uname = foo[2:end].split("\\\\")[1]
        datapath = "C:\\Users\\" + uname + "\\Documents\\AGCM_Experiments\\" + expname + "\\"
    case 'Darwin':
        foo = str(subprocess.check_output(['whoami']))
        end = len(foo) - 3
        uname = foo[2:end]
        datapath = '/Users/' + uname + '/Documents/AGCM_Experiments/' + expname + '/'
    case 'Linux':
        foo = str(subprocess.check_output(['whoami']))
        end = len(foo) - 3
        uname = foo[2:end]
        datapath = '/home/'+uname+'/projects/Atmospheric-Teleconnection-Model-main/'+expname+'/'

    case _:
        raise Exception("Use case for this system/OS is not implemented. Consider using custom_path in the advanced variables.")

# Set stamp for file names
stamp = 'days_1-' + str(dayst)

print("datapath =", datapath,
      "\nstamp =", stamp)

Setting output datapath for Linux
datapath = /home/kpegion/projects/Atmospheric-Teleconnection-Model-main/T63L26_ANA/ 
stamp = days_1-720


In [5]:
fps = datapath+'lnps_1*.nc' # always need surface pressure
dps = xarray.open_mfdataset(fps,decode_times=True,parallel = True)
#
#
fdata = datapath+DataSetname+'_1*.nc'
ddata = xarray.open_mfdataset(fdata,decode_times=True,parallel = True)

In [6]:
fps

'/home/kpegion/projects/Atmospheric-Teleconnection-Model-main/T63L26_ANA/lnps_1*.nc'

In [7]:
dps

<xarray.Dataset>
Dimensions:  (time: 720, lat: 96, lon: 192)
Coordinates:
  * time     (time) datetime64[ns] 1950-01-01 1950-01-02 ... 1951-12-21
  * lat      (lat) float64 88.57 86.72 84.86 83.0 ... -83.0 -84.86 -86.72 -88.57
  * lon      (lon) float64 0.0 1.875 3.75 5.625 7.5 ... 352.5 354.4 356.2 358.1
Data variables:
    lnps     (time, lat, lon) float64 dask.array<chunksize=(30, 96, 192), meta=np.ndarray>

In [8]:
ddata

<xarray.Dataset>
Dimensions:  (time: 720, lev: 26, lat: 96, lon: 192)
Coordinates:
  * time     (time) datetime64[ns] 1950-01-01 1950-01-02 ... 1951-12-21
  * lev      (lev) float64 0.001123 0.005056 0.01163 ... 0.9294 0.9705 0.9925
  * lat      (lat) float64 88.57 86.72 84.86 83.0 ... -83.0 -84.86 -86.72 -88.57
  * lon      (lon) float64 0.0 1.875 3.75 5.625 7.5 ... 352.5 354.4 356.2 358.1
Data variables:
    geo      (time, lev, lat, lon) float64 dask.array<chunksize=(30, 26, 96, 192), meta=np.ndarray>

In [9]:
#
# Create Data Array for Control Pressure level Data geopotenial, temp, u & v
# 
#
lats = ddata['lat'].values
lons = ddata['lon'].values
#plev = [1000.0,900.0,800.0,700.0,600.0,500.0,400.0,300.0,200.0,100.0,20.0]
plev = [850.0,500.0,300.0,200.0]

#plev_r = np.zeros(11)
plev_r = np.zeros(len(plev))
for k in range(len(plev)):
#for k in range(11):
    plev_r[k] = (plev[k])*100.0 # mb to Pa
#
#
#tmp = (dayst,11,jmax,imax) #### "11" here corresponds to standard pressure levels not to model levels-- should not be hardcoded!!!!
tmp = (dayst,len(plev),jmax,imax) #### "11" here corresponds to standard pressure levels not to model levels -- should not be hardcoded!!!

dout = np.zeros(tmp)
pressure = np.zeros((kmax,jmax,imax))
siglevs = ddata['lev']
for k in range (dayst):
    vv = ddata[Dataname][k,:,:,:]
    ps = dps.lnps[k,:,:]
    surfp = (np.exp(ps))*1000.0*100.0 # in Pa
    for kk in range(kmax):
        pressure[kk,:,:] = surfp[:,:]*siglevs[kk]
    vv = vv.compute()
    ps = ps.compute()
    dout[k] = metpy.interpolate.log_interpolate_1d(plev_r,pressure,vv, axis=0)
#
times = ddata['time']
dData = xarray.Dataset({Dataname: (['time','lev','lat','lon'],dout)},
                        coords={'time': times,'lev':plev, 'lat': lats, 'lon': lons})

/tmp/ipykernel_1045346/832001984.py:31: UserWarning: Interpolation point out of data bounds encountered
  dout[k] = metpy.interpolate.log_interpolate_1d(plev_r,pressure,vv, axis=0)
/tmp/ipykernel_1045346/832001984.py:31: UserWarning: Interpolation point out of data bounds encountered
  dout[k] = metpy.interpolate.log_interpolate_1d(plev_r,pressure,vv, axis=0)
/tmp/ipykernel_1045346/832001984.py:31: UserWarning: Interpolation point out of data bounds encountered
  dout[k] = metpy.interpolate.log_interpolate_1d(plev_r,pressure,vv, axis=0)
/tmp/ipykernel_1045346/832001984.py:31: UserWarning: Interpolation point out of data bounds encountered
  dout[k] = metpy.interpolate.log_interpolate_1d(plev_r,pressure,vv, axis=0)
/tmp/ipykernel_1045346/832001984.py:31: UserWarning: Interpolation point out of data bounds encountered
  dout[k] = metpy.interpolate.log_interpolate_1d(plev_r,pressure,vv, axis=0)
/tmp/ipykernel_1045346/832001984.py:31: UserWarning: Interpolation point out of data bounds enc

In [10]:
dout.shape

(720, 4, 96, 192)

In [11]:
dData

<xarray.Dataset>
Dimensions:  (time: 720, lev: 4, lat: 96, lon: 192)
Coordinates:
  * time     (time) datetime64[ns] 1950-01-01 1950-01-02 ... 1951-12-21
  * lev      (lev) float64 850.0 500.0 300.0 200.0
  * lat      (lat) float64 88.57 86.72 84.86 83.0 ... -83.0 -84.86 -86.72 -88.57
  * lon      (lon) float64 0.0 1.875 3.75 5.625 7.5 ... 352.5 354.4 356.2 358.1
Data variables:
    geo      (time, lev, lat, lon) float64 1.45e+04 1.45e+04 ... 1.119e+05

In [12]:
dData.to_netcdf(datapath+DataSetname+'_Pressure_'+stamp+'.nc')